## Gaussian Naive Bayes

In [1]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

In [4]:
iris = load_iris()

X = iris.data
y = iris.target

print("Input shape:", X.shape)
print("Output shape:", y.shape)

print("Feature names:", iris.feature_names)
print("Target names:", iris.target_names)

Input shape: (150, 4)
Output shape: (150,)
Feature names: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Target names: ['setosa' 'versicolor' 'virginica']


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])

Training records: 120
Testing records: 30


In [6]:
model = GaussianNB()

In [7]:
model.fit(X_train, y_train)

GaussianNB()

In [8]:
y_pred = model.predict(X_test)

print("Actual values:    ", y_test[:10])
print("Predicted values: ", y_pred[:10])

Actual values:     [0 2 1 1 0 1 0 0 2 1]
Predicted values:  [0 2 1 1 0 1 0 0 2 1]


In [9]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("Accuracy percentage:", accuracy * 100, "%")

Accuracy: 0.9666666666666667
Accuracy percentage: 96.66666666666667 %


In [10]:
cm = confusion_matrix(y_test, y_pred)

print(cm)

[[10  0  0]
 [ 0  9  1]
 [ 0  0 10]]


In [11]:
print(
    classification_report(
        y_test,
        y_pred,
        target_names=iris.target_names
    )
)

              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      0.90      0.95        10
   virginica       0.91      1.00      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30



In [12]:
new_flower = np.array([[5.8, 2.7, 4.1, 1.0]])

prediction = model.predict(new_flower)

predicted_species = iris.target_names[prediction[0]]

print("Predicted species:", predicted_species)

Predicted species: versicolor


In [13]:
probabilities = model.predict_proba(new_flower)

for species, probability in zip(
    iris.target_names,
    probabilities[0]
):
    print(f"{species}: {probability:.4f}")

setosa: 0.0000
versicolor: 1.0000
virginica: 0.0000


## Multinomial Naive Bayes

In [14]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

In [15]:
df = pd.read_csv("spam.csv", encoding="latin-1")

df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [16]:
df = df[['v1', 'v2']]

df.columns = ['label', 'message']

df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [17]:
print(df.shape)

print(df.isnull().sum())

print(df['label'].value_counts())

(5572, 2)
label      0
message    0
dtype: int64
label
ham     4825
spam     747
Name: count, dtype: int64


In [18]:
df = df.dropna(subset=['label', 'message'])

df = df.drop_duplicates()

df = df.reset_index(drop=True)

print(df.shape)

(5169, 2)


In [19]:
X = df['message']
y = df['label']

print("Input:")
print(X.head())

print("\nOutput:")
print(y.head())

Input:
0    Go until jurong point, crazy.. Available only ...
1                        Ok lar... Joking wif u oni...
2    Free entry in 2 a wkly comp to win FA Cup fina...
3    U dun say so early hor... U c already then say...
4    Nah I don't think he goes to usf, he lives aro...
Name: message, dtype: object

Output:
0     ham
1     ham
2    spam
3     ham
4     ham
Name: label, dtype: object


In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

Training records: 4135
Testing records: 1034


In [21]:
vectorizer = CountVectorizer()

X_train_count = vectorizer.fit_transform(X_train)

X_test_count = vectorizer.transform(X_test)

'''Understand the two important functions:
- fit_transform() learns the vocabulary from the training messages and converts them into word-count features.
- transform() converts test messages using the same vocabulary.
For example, the message "free prize free" could be represented with a count of 2 for free and 1 for prize.'''

print("Training shape:", X_train_count.shape)
print("Testing shape:", X_test_count.shape)

Training shape: (4135, 7591)
Testing shape: (1034, 7591)


In [22]:
model = MultinomialNB()

model.fit(X_train_count, y_train)

MultinomialNB()

In [23]:
y_pred = model.predict(X_test_count)

print(y_pred[:10])

['ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham' 'ham']


In [24]:
comparison = pd.DataFrame({
    'Actual': y_test.iloc[:10].values,
    'Predicted': y_pred[:10]
})

print(comparison)

  Actual Predicted
0    ham       ham
1    ham       ham
2    ham       ham
3    ham       ham
4    ham       ham
5    ham       ham
6    ham       ham
7    ham       ham
8    ham       ham
9    ham       ham


In [25]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("Accuracy percentage:", accuracy * 100, "%")

Accuracy: 0.9864603481624759
Accuracy percentage: 98.64603481624758 %


In [26]:
cm = confusion_matrix(
    y_test,
    y_pred,
    labels=['ham', 'spam']
)

print(cm)

[[900   3]
 [ 11 120]]


In [27]:
print(
    classification_report(
        y_test,
        y_pred,
        labels=['ham', 'spam'],
        zero_division=0
    )
)

              precision    recall  f1-score   support

         ham       0.99      1.00      0.99       903
        spam       0.98      0.92      0.94       131

    accuracy                           0.99      1034
   macro avg       0.98      0.96      0.97      1034
weighted avg       0.99      0.99      0.99      1034



In [28]:
new_message = [
    "Congratulations! You have won a free prize. Claim your reward now"
]

new_message_count = vectorizer.transform(new_message)

prediction = model.predict(new_message_count)

print("Message:", new_message[0])
print("Prediction:", prediction[0])

Message: Congratulations! You have won a free prize. Claim your reward now
Prediction: spam


In [29]:
probabilities = model.predict_proba(new_message_count)

for label, probability in zip(model.classes_, probabilities[0]):
    print(f"{label}: {probability:.4f}")

ham: 0.0000
spam: 1.0000


## Bernoulli Naive Bayes

In [30]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import BernoulliNB

from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

In [31]:
df = pd.read_csv("spam.csv", encoding="latin-1")

df = df[['v1', 'v2']]

df.columns = ['label', 'message']

df = df.dropna(subset=['label', 'message'])
df = df.drop_duplicates()
df = df.reset_index(drop=True)

df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [32]:
X = df['message']
y = df['label']

In [33]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

Training records: 4135
Testing records: 1034


In [35]:
vectorizer = CountVectorizer(binary=True)

'''
What does binary=True do?
It converts every word count into a binary value:
- 1 means the word is present.
- 0 means the word is absent.
'''

X_train_binary = vectorizer.fit_transform(X_train)

X_test_binary = vectorizer.transform(X_test)

print("Training shape:", X_train_binary.shape)
print("Testing shape:", X_test_binary.shape)

Training shape: (4135, 7591)
Testing shape: (1034, 7591)


In [36]:
model = BernoulliNB()

model.fit(X_train_binary, y_train)

BernoulliNB()

In [37]:
y_pred = model.predict(X_test_binary)

comparison = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': y_pred
})

print(comparison.head(10))

  Actual Predicted
0    ham       ham
1    ham       ham
2    ham       ham
3    ham       ham
4    ham       ham
5    ham       ham
6    ham       ham
7    ham       ham
8    ham       ham
9    ham       ham


In [38]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("Accuracy percentage:", accuracy * 100, "%")

Accuracy: 0.9758220502901354
Accuracy percentage: 97.58220502901354 %


In [39]:
cm = confusion_matrix(
    y_test,
    y_pred,
    labels=['ham', 'spam']
)

print(cm)

[[902   1]
 [ 24 107]]


In [40]:
print(
    classification_report(
        y_test,
        y_pred,
        labels=['ham', 'spam'],
        zero_division=0
    )
)

              precision    recall  f1-score   support

         ham       0.97      1.00      0.99       903
        spam       0.99      0.82      0.90       131

    accuracy                           0.98      1034
   macro avg       0.98      0.91      0.94      1034
weighted avg       0.98      0.98      0.97      1034



In [41]:
new_message = [
    "Congratulations! You have won a free prize. Claim your reward now"
]

new_message_binary = vectorizer.transform(new_message)

prediction = model.predict(new_message_binary)

print("Message:", new_message[0])
print("Prediction:", prediction[0])

Message: Congratulations! You have won a free prize. Claim your reward now
Prediction: spam


In [42]:
probabilities = model.predict_proba(new_message_binary)

for label, probability in zip(model.classes_, probabilities[0]):
    print(f"{label}: {probability:.4f}")

ham: 0.0024
spam: 0.9976


## Categorical Naive Bayes

In [43]:
# Step 1: Import the required libraries

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.naive_bayes import CategoricalNB
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

In [44]:
# Step 2: Load the Excel dataset

df = pd.read_excel("weather.xlsx")

print("First 5 rows:")
print(df.head())

print("\nDataset shape:")
print(df.shape)

print("\nColumn names:")
print(df.columns)


First 5 rows:
   Day   Outlook Temperature Humidity    Wind Play
0    1     Sunny         Hot     High    Weak   No
1    2     Sunny         Hot     High  Strong   No
2    3  Overcast         Hot     High    Weak  Yes
3    4      Rain        Mild     High    Weak  Yes
4    5      Rain        Cool   Normal    Weak  Yes

Dataset shape:
(14, 6)

Column names:
Index(['Day', 'Outlook', 'Temperature', 'Humidity', 'Wind', 'Play'], dtype='object')


In [45]:
# Step 3: Remove the Day column
# Day is only a record number; it is not a weather feature.

df = df.drop(columns=["Day"])

In [46]:
# Step 4: Separate input features (X) and target (y)

X = df[["Outlook", "Temperature", "Humidity", "Wind"]]
y = df["Play"]

In [47]:
# Step 5: Split the data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

In [48]:
# Step 6: Convert categorical input values into numbers

encoder = OrdinalEncoder()

X_train_encoded = encoder.fit_transform(X_train).astype(int)
X_test_encoded = encoder.transform(X_test).astype(int)

In [49]:
# Step 7: Convert the target labels into numbers

target_encoder = LabelEncoder()

y_train_encoded = target_encoder.fit_transform(y_train)
y_test_encoded = target_encoder.transform(y_test)

In [50]:
# Step 8: Create the Categorical Naive Bayes model

model = CategoricalNB(alpha=1.0)

'''CategoricalNB() creates the model.
alpha=1.0 applies Laplace smoothing to avoid zero conditional probabilities for categories.
fit() learns the class probabilities and categorical feature probabilities from the training data.'''

In [51]:
# Step 9: Train the model

model.fit(X_train_encoded, y_train_encoded)

CategoricalNB()

In [52]:
# Step 10: Predict the test data

y_pred = model.predict(X_test_encoded)


In [53]:
# Step 11: Evaluate the model

print("\nAccuracy:")
print(accuracy_score(y_test_encoded, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_encoded, y_pred))

print("\nClassification Report:")
print(
    classification_report(
        y_test_encoded,
        y_pred,
        target_names=target_encoder.classes_,
        zero_division=0
    )
)




Accuracy:
0.6

Confusion Matrix:
[[0 2]
 [0 3]]

Classification Report:
              precision    recall  f1-score   support

          No       0.00      0.00      0.00         2
         Yes       0.60      1.00      0.75         3

    accuracy                           0.60         5
   macro avg       0.30      0.50      0.38         5
weighted avg       0.36      0.60      0.45         5



In [54]:
# Step 12: Predict for a new weather condition

new_weather = pd.DataFrame({
    "Outlook": ["Overcast"],
    "Temperature": ["Hot"],
    "Humidity": ["Normal"],
    "Wind": ["Strong"]
})

# Use the same encoder that was fitted on training data
new_weather_encoded = encoder.transform(new_weather).astype(int)

# Predict
prediction = model.predict(new_weather_encoded)

# Convert numeric prediction back to Yes/No
predicted_play = target_encoder.inverse_transform(prediction)

print("\nNew Weather Condition:")
print(new_weather)

print("\nPredicted Play:", predicted_play[0])


New Weather Condition:
    Outlook Temperature Humidity    Wind
0  Overcast         Hot   Normal  Strong

Predicted Play: Yes


In [55]:
# Step 13: Display prediction probabilities

probabilities = model.predict_proba(new_weather_encoded)[0]

print("\nPrediction Probabilities:")

for label, probability in zip(target_encoder.classes_, probabilities):
    print(f"{label}: {probability:.4f}")



Prediction Probabilities:
No: 0.3386
Yes: 0.6614
